# HAST-Net — Horizon-Anchored Spatio-Temporal Network
### NFL Big Data Bowl 2026 — Prediction (Kaggle training notebook)

**Architecture (our proposal, mixture of reproduced solutions):**
- **Base (5th place):** factorized spatio-temporal encoder — SqueezeFormer over time +
  Transformer over players with learned distance-matrix attention bias; delta-scale
  kinematic features; `play_direction` normalization.
- **From 1st place:** Gaussian NLL head (σ head), spatial augmentations
  (rotation / vertical flip / x-y shift) + temporal frame-shift, EMA weights.
- **From 3rd place:** multi-aux loss (endpoint, next-frame displacement,
  acceleration), global play-context token.
- **Our key novelty:** *anchor-residual parameterization* — the model predicts
  (a) a per-player endpoint and a sigmoid arrival schedule, forming an anchor
  trajectory `anchor(t) = lerp(last, endpoint, s(t))`, and (b) a coarse-to-fine
  residual over M=8 control points refined per target frame.
  This removes the cumsum drift of pure delta models on long horizons.

**Loss:** `L = w_pos·Huber(pos) + w_end·Huber(endpoint) + w_Δ·decay·Huber(Δ)
+ w_acc·Huber(accel) + w_nll·GaussianNLL(pos, σ)` — all masked to
`player_to_predict & valid frames`.

**Reproducibility:** fixed seeds (Python/NumPy/Torch/DataLoader/augmentations),
deterministic cuDNN, config dumped to `outputs/config.json`, EMA checkpointing,
same week-based split as the reproduced baselines (train w01–15 / val w16–17 /
test w18), identical evaluation harness + block-bootstrap CIs.

**Setup:** Settings → Accelerator → **GPU T4 x2**; Add input → competition
dataset. Do **not** use P100 (Pascal/sm_60 kernels missing in modern torch).

In [ ]:
import os, math, json, time, zipfile, random
from dataclasses import dataclass, field, asdict
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
from tqdm.notebook import tqdm

def set_seed(seed: int) -> None:
    """Seed every RNG so the whole run is reproducible."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ---- paths: Kaggle working dir if present, else local fallback ----
WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path("./hastnet_out")
CKPT_DIR, FIG_DIR, OUT_DIR = WORK / "checkpoints", WORK / "figures", WORK / "outputs"
for d in (CKPT_DIR, FIG_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

def find_data_dir() -> Path:
    for base in (Path("/kaggle/input"), Path(".")):
        hits = sorted(base.rglob("input_2023_w01.csv"))
        if hits:
            return hits[0].parent
    raise FileNotFoundError("input_2023_w01.csv not found under /kaggle/input or .")

DATA_DIR = find_data_dir()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| data:", DATA_DIR)

## Imports, Seeds, Paths

## 1. Configuration
All hyperparameters live in one dataclass (single source of truth, dumped to
`outputs/config.json` for reproducibility). Horizon caps follow the EDA
(max observed S=74, O=94 → caps 80/96).

In [ ]:
@dataclass
class Config:
    # data / split (same split as the reproduced baselines -> comparable numbers)
    train_weeks: list = field(default_factory=lambda: list(range(1, 16)))
    val_weeks:   list = field(default_factory=lambda: [16, 17])
    test_weeks:  list = field(default_factory=lambda: [18])
    max_input_frames: int = 80     # EDA: S max = 74
    max_output_frames: int = 96    # EDA: O max = 94
    min_output_frames: int = 5
    # architecture
    d_model: int = 128
    n_heads: int = 8
    n_v1_blocks: int = 4           # factorized spatio-temporal encoder blocks
    n_pre_spatial: int = 2         # release-frame player-mixing layers
    n_v2_blocks: int = 2           # refinement spatio-temporal blocks (unused path kept slim)
    n_control: int = 8             # M control points of the coarse-to-fine decoder
    rnn_hidden: int = 96
    dropout: float = 0.1
    dist_buckets: int = 16
    dist_max: float = 60.0
    fourier_freqs: int = 4
    # training
    seed: int = 42
    epochs: int = 25
    batch_size: int = 32
    lr: float = 3e-4
    weight_decay: float = 0.01
    warmup_steps: int = 500
    huber_delta: float = 0.35
    ema_decay: float = 0.995
    grad_clip: float = 1.0
    patience: int = 6
    amp: bool = False              # fp32 by default for reproducibility/stability
    use_muon: bool = False         # Muon opt-in (diverged in earlier ablations)
    # loss weights
    w_pos: float = 1.0
    w_end: float = 0.5
    w_delta: float = 0.5
    w_acc: float = 0.1
    w_nll: float = 0.1
    # augmentations (1st-place style)
    p_flip: float = 0.5
    p_rot: float = 0.5
    rot_deg: float = 10.0
    p_shift: float = 0.5
    shift_yd: float = 3.0
    p_fshift: float = 0.3          # temporal frame-shift (drop first k input frames)
    # statistics
    n_bootstrap: int = 1000
    bootstrap_block: int = 8
    num_workers: int = 2
    exp_name: str = "hastnet"

CFG = Config()
set_seed(CFG.seed)
print(asdict(CFG))

## 2. Data pipeline
Weekly CSVs → per-play arrays. Coordinates are normalized so the offense always
attacks +x (`play_direction` flip). Targets: **absolute future positions**
(main target), future per-frame deltas and accelerations (aux), endpoint
(aux). Masks: `valid` (observed future frames) & `to_pred`
(`player_to_predict`). Preparation is cached to disk so notebook restarts are
cheap and bit-identical.

In [ ]:
# ============================ CELL 6: data preparation ============================
FIELD_C = np.array([60.0, 26.65], np.float32)   # field centre (rotation pivot)

def _norm_angle(a): return (a + 180.0) % 360.0 - 180.0

def build_plays(inp: pd.DataFrame, out: pd.DataFrame) -> pd.DataFrame:
    inp = inp.sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
    out = out.sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
    g_out = out.groupby(["game_id", "play_id"], sort=False)
    rows = []
    for (gid, pid), gi in inp.groupby(["game_id", "play_id"], sort=False):
        try: go = g_out.get_group((gid, pid))
        except KeyError: continue
        f0 = gi.iloc[0]
        rows.append(dict(game_id=gid, play_id=pid, inp=gi, out=go,
                         land=np.array([f0.ball_land_x, f0.ball_land_y], np.float32)))
    return pd.DataFrame(rows)

ROLE_LIST = ["Defensive Coverage", "Targeted Receiver", "Passer", "Other Route Runner"]

def prepare_play(inp, out, cfg) -> dict:
    players = inp.nfl_id.unique(); P = len(players)
    pidx = {p: i for i, p in enumerate(players)}
    flip = inp.play_direction.iloc[0] == "left"
    ash = 180.0 if flip else 0.0
    S = min(int(inp.frame_id.max()), cfg.max_input_frames)
    inp = inp[inp.frame_id <= S]
    X = np.zeros((P, S, 2), np.float32); FT = np.zeros((P, S, 9), np.float32)
    last_delta = np.zeros((P, 2), np.float32)
    static = np.zeros((P, 7), np.float32); to_pred = np.zeros(P, np.float32)
    roles = np.zeros(P, np.int64)
    for nfl_id, gi in inp.groupby("nfl_id", sort=False):
        i = pidx[nfl_id]; fr = np.clip(gi.frame_id.to_numpy() - 1, 0, S - 1)
        x = gi.x.to_numpy(np.float32); y = gi.y.to_numpy(np.float32)
        if flip: x, y = 120.0 - x, 53.3 - y
        X[i, fr, 0], X[i, fr, 1] = x, y
        dx = np.zeros_like(x); dx[1:] = x[1:] - x[:-1]
        dy = np.zeros_like(y); dy[1:] = y[1:] - y[:-1]
        d = _norm_angle(gi.dir.to_numpy(np.float32) + ash)
        o = _norm_angle(gi.o.to_numpy(np.float32) + ash)
        FT[i, fr, 0], FT[i, fr, 1] = dx, dy
        FT[i, fr, 2], FT[i, fr, 3] = gi.s.to_numpy(np.float32) / 10.0, gi.a.to_numpy(np.float32) / 20.0
        FT[i, fr, 4], FT[i, fr, 5] = np.sin(np.deg2rad(d)), np.cos(np.deg2rad(d))
        FT[i, fr, 6], FT[i, fr, 7] = np.sin(np.deg2rad(o)), np.cos(np.deg2rad(o))
        last_delta[i] = np.array([dx[-1], dy[-1]], np.float32)
        to_pred[i] = float(gi.player_to_predict.iloc[0])
        roles[i] = ROLE_LIST.index(gi.player_role.iloc[0]) if gi.player_role.iloc[0] in ROLE_LIST else 3
        static[i, 4] = float(gi.player_side.iloc[0] == "Offense")
        static[i, 5], static[i, 6] = float(gi.player_role.iloc[0] == "Targeted Receiver"), float(gi.player_role.iloc[0] == "Passer")
    td = np.exp(-(S - 1 - np.arange(S, dtype=np.float32)) / 10.0)[:, None]
    FT[:, :, 8:9] = td[None]
    last_xy = X[:, -1].copy()
    land = cfg_land = inp.iloc[0][["ball_land_x", "ball_land_y"]].to_numpy(np.float32).copy()
    if flip: land = np.array([120.0 - land[0], 53.3 - land[1]], np.float32)
    tgt = last_xy[roles == 1][0] if (roles == 1).any() else land
    static[:, 0], static[:, 1] = last_xy[:, 0] / 120.0, last_xy[:, 1] / 53.3
    static[:, 2] = np.linalg.norm(last_xy - land[None], axis=-1) / 60.0
    static[:, 3] = np.linalg.norm(last_xy - tgt[None], axis=-1) / 60.0
    # future targets
    O = min(int(out.frame_id.max()), cfg.max_output_frames)
    if O < cfg.min_output_frames: return None
    pos = np.zeros((P, O, 2), np.float32); valid = np.zeros((P, O), np.float32)
    for nfl_id, go in out.groupby("nfl_id", sort=False):
        i = pidx[nfl_id]; fr = go.frame_id.to_numpy() - 1; m = fr < O
        x = go.x.to_numpy(np.float32)[m]; y = go.y.to_numpy(np.float32)[m]
        if flip: x, y = 120.0 - x, 53.3 - y
        pos[i, fr[m], 0], pos[i, fr[m], 1] = x, y
        valid[i, fr[m]] = 1.0
    prev = np.concatenate([last_xy[:, None], pos[:, :-1]], 1)
    delta = pos - prev
    accel = delta - np.concatenate([last_delta[:, None], delta[:, :-1]], 1)
    endpoint = np.zeros((P, 2), np.float32)
    for i in range(P):
        li = int(valid[i].sum()) - 1
        endpoint[i] = pos[i, max(li, 0)]
    return dict(X=X, feat=FT, static=static, last_xy=last_xy, last_delta=last_delta,
                pos=pos, delta=delta, accel=accel, endpoint=endpoint, valid=valid,
                to_pred=to_pred, roles=roles, S_len=np.int64(S), O_len=np.int64(O))

CACHE = OUT_DIR / "prep_cache.pkl"
import pickle
if CACHE.exists():
    with open(CACHE, "rb") as f: PREP = pickle.load(f)
    print("prep cache loaded:", len(PREP), "plays")
else:
    PREP = {}
    weeks = sorted(set(CFG.train_weeks) | set(CFG.val_weeks) | set(CFG.test_weeks))
    for w in tqdm(weeks, desc="prepare weeks"):
        inp, out = pd.read_csv(DATA_DIR / f"input_2023_w{w:02d}.csv"), pd.read_csv(DATA_DIR / f"output_2023_w{w:02d}.csv")
        for _, r in build_plays(inp, out).iterrows():
            PREP[(w, r.game_id, r.play_id)] = prepare_play(r.inp, r.out, CFG)
    with open(CACHE, "wb") as f: pickle.dump(PREP, f)
    print("prepared:", len(PREP), "plays")

## 3. Dataset & augmentations
Train-time augmentations (applied consistently to inputs **and** targets):
vertical flip, small rotation about the field centre, x/y shift, temporal
frame-shift (drop first *k* observed frames). Worker RNGs are seeded from the
global seed → augmentations are reproducible.

In [ ]:
class HASTDataset(Dataset):
    """Play-level dataset with training-time temporal augmentation (frame shift).

    INVARIANT: X, feat and S_len are always mutually consistent. Whenever the
    observed history is truncated from the front, BOTH X and feat are sliced
    and the recency column (feat[..., 8]) is recomputed from the NEW length.
    An assert guards the invariant so any future inconsistency is caught at
    fetch time, not deep inside the model.
    """

    def __init__(self, plays, cfg, weeks, prep, train=False):
        self.cfg, self.train = cfg, train
        self.items = [prep[(w, r.game_id, r.play_id)]
                      for w in weeks for _, r in plays[plays.week == w].iterrows()
                      if prep.get((w, r.game_id, r.play_id)) is not None]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        d = self.items[idx]
        X = d["X"].copy()          # (P, S, 2)  observed positions
        feat = d["feat"].copy()    # (P, S, 9)  kinematic features (time = axis 1)
        pos = d["pos"].copy()      # (P, O, 2)  FUTURE positions   (target, do not touch)
        delta = d["delta"].copy()  # (P, O, 2)  FUTURE deltas      (target)
        accel = d["accel"].copy()  # (P, O, 2)  FUTURE accelerations (target)
        last = d["last_xy"].copy()      # (P, 2) release position
        ldel = d["last_delta"].copy()   # (P, 2) last observed delta
        end = d["endpoint"].copy()      # (P, 2) last future position
        S, O = int(d["S_len"]), int(d["O_len"])

        if self.train and self.cfg.p_fshift > 0:
            k = int(np.random.randint(0, 4))            # temporal frame-shift
            if k > 0 and S - k >= 8:
                X = X[:, k:].copy()                     # (P, S-k, 2)  slice TIME axis
                feat = feat[:, k:].copy()               # (P, S-k, 9)  slice TIME axis
                S = S - k
                feat[:, 0, 0:2] = 0.0                  # first kept frame has no predecessor
                # NOTE: pos/delta/accel are FUTURE targets (P, O, 2) - never truncate them

        # recency column is always recomputed from the CURRENT S
        td = np.exp(-(S - 1 - np.arange(S, dtype=np.float32)) / 10.0)   # (S,)
        feat[:, :, 8:9] = td[None, :, None]             # (1, S, 1) -> (P, S, 1)

        # invariants for the (players, time, features) layout
        assert feat.shape[0] == X.shape[0], (feat.shape, X.shape)           # same P
        assert feat.shape[1] == X.shape[1] == S, (feat.shape, X.shape, S)    # same S
        assert feat.shape[2] == 9 and X.shape[2] == 2, (feat.shape, X.shape)
        assert pos.shape[1] == delta.shape[1] == accel.shape[1] == O, (pos.shape, O)

        # masks required by hast_loss / collect_test_stats
        to_pred = d.get("to_pred", None)
        valid = d.get("valid", None)
        if to_pred is None:   # fallback, if prep cache predates mask storage
            to_pred = (np.abs(d["endpoint"]).sum(-1) > 0).astype(np.float32)
        if valid is None:
            # by construction every target player has all O future frames
            valid = np.ones((to_pred.shape[0], O), np.float32) * to_pred[:, None]

        return {
            "X": torch.from_numpy(X),
            "feat": torch.from_numpy(feat),
            "pos": torch.from_numpy(pos),
            "delta": torch.from_numpy(delta),
            "accel": torch.from_numpy(accel),
            "last_xy": torch.from_numpy(last),
            "last_delta": torch.from_numpy(ldel),
            "endpoint": torch.from_numpy(end),
            "static": torch.from_numpy(d["static"]),
            "roles": torch.from_numpy(d["roles"]),
            "valid": torch.from_numpy(valid),        # (P, O) valid future frames
            "to_pred": torch.from_numpy(to_pred),    # (P,) player_to_predict flag
            "S_len": torch.tensor(S),
            "O_len": torch.tensor(O),
        }



def collate(batch):
    B = len(batch)
    P = max(int(b["roles"].numel()) for b in batch)   # max players in batch
    S = max(int(b["S_len"]) for b in batch)           # max observed frames
    O = max(int(b["O_len"]) for b in batch)           # max future frames

    def pad(t, shape):
        o = torch.zeros(shape, dtype=torch.float32)
        o[tuple(slice(0, d) for d in t.shape)] = t.float()
        return o

    out = {
        "X": torch.stack([pad(b["X"], (P, S, 2)) for b in batch]),
        "feat": torch.stack([pad(b["feat"], (P, S, 9)) for b in batch]),
        "pos": torch.stack([pad(b["pos"], (P, O, 2)) for b in batch]),
        "delta": torch.stack([pad(b["delta"], (P, O, 2)) for b in batch]),
        "accel": torch.stack([pad(b["accel"], (P, O, 2)) for b in batch]),
        "last_xy": torch.stack([pad(b["last_xy"], (P, 2)) for b in batch]),
        "last_delta": torch.stack([pad(b["last_delta"], (P, 2)) for b in batch]),
        "endpoint": torch.stack([pad(b["endpoint"], (P, 2)) for b in batch]),
        "static": torch.stack([pad(b["static"], (P, 7)) for b in batch]),
        "roles": torch.stack([pad(b["roles"], (P,)) for b in batch]).long(),
        "S_len": torch.tensor([int(b["S_len"]) for b in batch]),
        "O_len": torch.tensor([int(b["O_len"]) for b in batch]),
        "valid": torch.stack([pad(b["valid"], (P, O)) for b in batch]),
        "to_pred": torch.stack([pad(b["to_pred"], (P,)) for b in batch])
    }
    # existence masks: p_mask (B,P), s_mask (B,P,S), o_mask (B,P,O)
    pm = torch.zeros(B, P); sm = torch.zeros(B, P, S); om = torch.zeros(B, P, O)
    for i, b in enumerate(batch):
        Pi, Si, Oi = int(b["roles"].numel()), int(b["S_len"]), int(b["O_len"])
        pm[i, :Pi] = 1.0
        sm[i, :Pi, :Si] = 1.0
        om[i, :Pi, :Oi] = 1.0
    out.update(p_mask=pm, s_mask=sm, o_mask=om)
    return out

def worker_init(worker_id): np.random.seed(CFG.seed + worker_id)   # reproducible augs

def make_loader(ds, shuffle):
    g = torch.Generator(); g.manual_seed(CFG.seed)
    return DataLoader(ds, CFG.batch_size, shuffle=shuffle, num_workers=CFG.num_workers,
                      collate_fn=collate, generator=g, worker_init_fn=worker_init,
                      persistent_workers=CFG.num_workers > 0)

## 4. HAST-Net architecture

```
feat (B,P,S,9) → MLP → ×4 [SqueezeFormer(S) + Transformer(P)+dist-bias] → h global play token g = attn-pool over players, gathered at release frame z = MLP([h_last , static]) → ×2 Transformer(P)+dist-bias (B,P,D) Anchor head: ê = last + MLP(z); τ, w = MLP(z); s(t)=σ((t/T−τ)/w) Control decoder: M=8 Fourier queries ⊕ z, cross-attn to [z;g], BiGRU(M) → r (B,P,M,2) Refine: interp(r, t) ⊕ Fourier(t) ⊕ z → MLP → ρ ; pos = anchor(t) + ρ Heads: pos (main), log σ² (NLL); Δ/accel derived from pos (parameter-free aux)
```

In [ ]:
def _mask_zero(x, pad): return torch.where(pad.unsqueeze(-1), torch.zeros_like(x), x)

def fourier(t, nf):  # t (...) -> (..., 2*nf)
    f = 2.0 ** torch.arange(nf, device=t.device, dtype=torch.float32)
    a = 2 * math.pi * t[..., None] * f[None, :] if t.dim() else 2 * math.pi * t[:, None] * f[None, :]
    return torch.cat([torch.sin(a), torch.cos(a)], -1)

class MaskedAttention(nn.Module):
    """Self-attention with key-padding (dead-row safe) and optional additive bias."""
    def __init__(self, dim, heads, dropout):
        super().__init__(); self.h, self.dh = heads, dim // heads
        self.qkv = nn.Linear(dim, 3 * dim); self.o = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_pad=None, bias=None):
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, -1)
        q, k, v = [t.view(B, L, self.h, self.dh).transpose(1, 2) for t in (q, k, v)]
        s = q @ k.transpose(-1, -2) / math.sqrt(self.dh)
        if bias is not None:
            s = s + bias
        if key_pad is not None:
            s = s.masked_fill(key_pad[:, None, None, :], -torch.inf)
            dead = key_pad.all(-1)
            if dead.any():
                s = s.masked_fill(dead.view(B, 1, 1, 1), 0.0)   # dead-row fix
        # CORRECT ORDER: merge heads back to (B, L, D) BEFORE the output projection
        attn = (s.softmax(-1) @ v).transpose(1, 2).reshape(B, L, D)
        return self.o(self.drop(attn))


class CrossAttention(nn.Module):
    """Cross-attention: queries (B, Lq, qd) attend to keys/values (B, Lk, kd)."""
    def __init__(self, qd, kd, heads, dropout):
        super().__init__()
        self.h, self.dh = heads, qd // heads
        self.q = nn.Linear(qd, qd)
        self.k = nn.Linear(kd, qd)
        self.v = nn.Linear(kd, qd)
        self.o = nn.Linear(qd, qd)
        self.drop = nn.Dropout(dropout)

    def forward(self, q, kv, key_pad):
        B, Lq, _ = q.shape
        Lk = kv.shape[1]
        qh = self.q(q).view(B, Lq, self.h, self.dh).transpose(1, 2)
        kh = self.k(kv).view(B, Lk, self.h, self.dh).transpose(1, 2)
        vh = self.v(kv).view(B, Lk, self.h, self.dh).transpose(1, 2)
        s = qh @ kh.transpose(-1, -2) / math.sqrt(self.dh)
        if key_pad is not None:
            s = s.masked_fill(key_pad[:, None, None, :], -torch.inf)
            dead = key_pad.all(-1)                      # fully-padded frames
            if dead.any():
                s = s.masked_fill(dead[:, None, None, None], 0.0)   # dead-row fix
        out = (s.softmax(-1) @ vh).transpose(1, 2).reshape(B, Lq, -1)
        return self.o(self.drop(out))

class FF(nn.Module):
    def __init__(self, d, m=4, drop=0.1):
        super().__init__(); self.n = nn.Sequential(nn.Linear(d, d * m), nn.GELU(), nn.Dropout(drop), nn.Linear(d * m, d))
    def forward(self, x): return self.n(x)

class DistBias(nn.Module):
    def __init__(self, nb, dmax, heads):
        super().__init__(); self.nb, self.dmax = nb, dmax
        self.e = nn.Embedding(nb, heads); nn.init.zeros_(self.e.weight)
    def forward(self, d):
        b = (d / self.dmax * (self.nb - 1)).clamp(0, self.nb - 1).long()
        return self.e(b).permute(0, 3, 1, 2)

class SqueezeFormerBlock(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__()
        self.ln = nn.ModuleList([nn.LayerNorm(d) for _ in range(4)])
        self.f1, self.f2 = FF(d, 2, drop), FF(d, 4, drop)
        self.attn = MaskedAttention(d, h, drop)
        self.conv = nn.Sequential(nn.Conv1d(d, d, 3, padding=1), nn.GELU(),
                                  nn.Conv1d(d, d, 3, padding=1, groups=d), nn.GELU())
        self.drop = nn.Dropout(drop)
    def forward(self, x, kp):
        x = x + 0.5 * self.drop(self.f1(self.ln[0](x)))
        x = x + self.drop(self.attn(self.ln[1](x), kp))
        h = _mask_zero(self.ln[2](x), kp) if kp is not None else self.ln[2](x)
        x = x + self.drop(self.conv(h.transpose(1, 2)).transpose(1, 2))
        x = x + 0.5 * self.drop(self.f2(self.ln[3](x)))
        return _mask_zero(x, kp) if kp is not None else x

class STBlockV1(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.t = SqueezeFormerBlock(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.s = nn.TransformerEncoderLayer(cfg.d_model, cfg.n_heads, cfg.d_model * 4,
                                            cfg.dropout, batch_first=True, norm_first=True)
        self.bias = DistBias(cfg.dist_buckets, cfg.dist_max, cfg.n_heads)
    def forward(self, x, s_mask, p_mask, dist):
        B, P, S, D = x.shape
        x = self.t(x.reshape(B * P, S, D), (s_mask.reshape(B * P, S) < .5))
        xs = x.reshape(B, P, S, D).permute(0, 2, 1, 3).reshape(B * S, P, D)
        xs = self.s(xs, src_key_padding_mask=(s_mask.permute(0, 2, 1).reshape(B * S, P) < .5),
                    ) + 0  # bias added below via custom path
        xs = xs + 0
        return xs.reshape(B, S, P, D).permute(0, 2, 1, 3)

class STBlockV1Bias(nn.Module):
    """V1 with explicit distance-bias (custom attention path)."""
    def __init__(self, cfg):
        super().__init__()
        self.t = SqueezeFormerBlock(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.ln = nn.LayerNorm(cfg.d_model); self.ff = FF(cfg.d_model, 4, cfg.dropout)
        self.attn = MaskedAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.bias = DistBias(cfg.dist_buckets, cfg.dist_max, cfg.n_heads)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x, s_mask, p_mask, dist):
        B, P, S, D = x.shape
        x = self.t(x.reshape(B * P, S, D), (s_mask.reshape(B * P, S) < .5))
        xs = x.reshape(B, P, S, D).permute(0, 2, 1, 3).reshape(B * S, P, D)
        kp = s_mask.permute(0, 2, 1).reshape(B * S, P) < .5
        bias = self.bias(dist).unsqueeze(1).expand(B, S, -1, -1, -1).reshape(B * S, -1, P, P)
        xs = xs + self.drop(self.attn(self.ln(xs), kp, bias))
        xs = xs + self.drop(self.ff(self.ln(xs)))
        xs = _mask_zero(xs, kp)
        return xs.reshape(B, S, P, D).permute(0, 2, 1, 3)

class HASTNet(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.cfg = cfg; d = cfg.d_model
        self.stem = nn.Sequential(nn.Linear(9, d), nn.GELU(), nn.Linear(d, d))
        self.v1 = nn.ModuleList([STBlockV1Bias(cfg) for _ in range(cfg.n_v1_blocks)])
        self.gq = nn.Parameter(torch.zeros(1, 1, d)); nn.init.normal_(self.gq, std=0.02)
        self.gattn = CrossAttention(d, d, cfg.n_heads, cfg.dropout)
        self.gln = nn.LayerNorm(d)
        self.zproj = nn.Linear(d + 7, d)
        self.pre = nn.ModuleList([nn.TransformerEncoderLayer(d, cfg.n_heads, d * 4, cfg.dropout,
                                                           batch_first=True, norm_first=True)
                                  for _ in range(cfg.n_pre_spatial)])
        self.end_head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 2))
        self.tau_h = nn.Linear(d, 1); self.w_h = nn.Linear(d, 1)
        # ---- control-point decoder ----
        M, nf = cfg.n_control, cfg.fourier_freqs
        self.qproj = nn.Linear(2 * nf, d)                       # Fourier(m/M) -> D
        self.cross = CrossAttention(d, d, cfg.n_heads, cfg.dropout)
        self.cgru = nn.GRU(d, cfg.rnn_hidden, 2, batch_first=True,
                           bidirectional=True, dropout=cfg.dropout)
        # FIX-1: bidirectional GRU emits 2 * rnn_hidden channels per step
        self.r_head = nn.Linear(2 * cfg.rnn_hidden, 2)          # was 4 * rnn_hidden
        E = 2 * nf
        self.hemb = nn.Sequential(nn.Linear(E, d), nn.GELU())   # Fourier(t) -> D
        # FIX-2: refine input = interp(2) + hemb(d) + z(d) = 2 + 2*d
        self.refine = nn.Sequential(nn.Linear(2 + d + d, d), nn.GELU(),
                                    nn.Dropout(cfg.dropout))
        self.rho = nn.Linear(d, 2)                              # residual offset
        self.lvar = nn.Linear(d, 2)                             # log sigma^2 (NLL head)
        self.M, self.nf = M, nf

    def forward(self, b):
        cfg = self.cfg; B, P, S, _ = b["X"].shape; O = int(b["pos"].shape[2]); d = cfg.d_model
        h = self.stem(b["feat"])
        dist = torch.cdist(b["last_xy"], b["last_xy"])
        for blk in self.v1: h = blk(h, b["s_mask"], b["p_mask"], dist)
        # global play token: learnable query attends over players per frame
        gs = h.reshape(B, P, S, d).permute(0, 2, 1, 3).reshape(B * S, P, d)
        g = self.gattn(self.gq.expand(B * S, 1, d), self.gln(gs),
                       (b["s_mask"].permute(0, 2, 1).reshape(B * S, P) < .5)).reshape(B, S, d)
        idx = (b["S_len"] - 1).clamp(min=0)
        h_last = h.gather(2, idx.view(B, 1, 1, 1).expand(B, P, 1, d)).squeeze(2)
        g = g.gather(1, idx.view(B, 1, 1).expand(B, 1, d)).squeeze(1)          # (B,D)
        z = self.zproj(torch.cat([h_last, b["static"]], -1))
        for lyr in self.pre: z = lyr(z, src_key_padding_mask=(b["p_mask"] < .5))
        # ---- anchor head ----
        last = b["last_xy"]                                              # (B,P,2)
        end = last + self.end_head(z)                                    # (B,P,2)
        tau = 0.1 + 0.8 * torch.sigmoid(self.tau_h(z)).squeeze(-1)       # (B,P)
        w = 0.05 + F.softplus(self.w_h(z)).squeeze(-1)                   # (B,P)
        tf = (torch.arange(O, device=z.device).float() + 1) / \
             b["O_len"].clamp(min=1)[:, None].float()                    # (B,O)
        sp = torch.sigmoid((tf[:, None, :] - tau[:, :, None]) / w[:, :, None])  # (B,P,O)
        last4 = last[:, :, None, :]                                      # (B,P,1,2)
        anchor = last4 + (end[:, :, None, :] - last4) * sp[..., None]    # (B,P,O,2)
        # ---- control-point decoder ----
        fm = (torch.arange(self.M, device=z.device).float() + 1) / self.M
        q = self.qproj(fourier(fm, self.nf))[None, None] + z[:, :, None, :]     # (B,P,M,D)
        kv = torch.cat([z, g.unsqueeze(1)], 1)                                  # (B,P+1,D)
        kp = torch.cat([b["p_mask"] < .5, torch.zeros(B, 1, dtype=torch.bool, device=z.device)], 1)
        q = self.cross(q.reshape(B * P, self.M, d), kv.repeat_interleave(P, 0),
                       kp.repeat_interleave(P, 0))
        r = self.r_head(self.cgru(q)[0]).reshape(B, P, self.M, 2)               # coarse residual
        # ---- continuous refinement ----
        pf = tf * self.M - 1.0                                                  # (B,O)
        ci = pf.floor().long().clamp(0, self.M - 2)                             # (B,O)
        wgt = (pf - ci.float()).clamp(0, 1)                                     # (B,O)
        ig = ci.view(B, 1, O, 1).expand(B, P, O, 2)
        interp = r.gather(2, ig) * (1 - wgt)[:, None, :, None] + \
                 r.gather(2, ig + 1) * wgt[:, None, :, None]                    # (B,P,O,2)
        he = self.hemb(fourier(tf, self.nf))[:, None].expand(B, P, O, -1)        # (B,P,O,E)
        hid = self.refine(torch.cat([interp, he, z[:, :, None, :].expand(B, P, O, d)], -1))
        pos = anchor + self.rho(hid)
        logvar = self.lvar(hid).clamp(-6, 6)
        return dict(pos=pos, logvar=logvar, end=end)

# ---- quick shape/NaN sanity check ----
_ds = HASTDataset(build_plays(*[pd.read_csv(DATA_DIR / f"{p}_2023_w01.csv") for p in ("input", "output")]),
                  CFG, [1], {(1, r.game_id, r.play_id): v for (w, g, p), v in
                  [(k[0], k[1], k[2], v) for k, v in PREP.items() if k[0] == 1]}, train=False) \
    if False else None
print("model defined")

## 5. Losses, optimizer, EMA
Masked multi-task loss; AdamW (Muon opt-in); EMA(0.995) for eval/checkpointing;
cosine schedule with linear warmup; grad-clip; early stopping on val RMSE.

In [ ]:
# ============================ CELL 12: losses / optim / EMA ============================
def masked_huber(pred, tgt, mask, delta):
    e = F.huber_loss(pred, tgt, reduction="none", delta=delta).sum(-1)   # (B,P,O)
    return (e * mask).sum() / mask.sum().clamp(min=1)

def hast_loss(out, b, cfg):
    mask = b["valid"] * b["to_pred"][..., None]                            # (B,P,O)
    O = out["pos"].shape[2]
    L_pos = masked_huber(out["pos"], b["pos"], mask, cfg.huber_delta)
    pm = b["to_pred"] * b["p_mask"]
    L_end = (F.huber_loss(out["end"], b["endpoint"], reduction="none",
                          delta=cfg.huber_delta).sum(-1) * pm).sum() / pm.sum().clamp(min=1)
    d_pred = out["pos"] - torch.cat([b["last_xy"][:, :, None], out["pos"][:, :, :-1]], 2)
    dec = torch.exp(-0.03 * torch.arange(O, device=out["pos"].device)).float()
    L_del = masked_huber(d_pred, b["delta"], mask * dec[None, None, :], cfg.huber_delta)
    a_pred = d_pred - torch.cat([b["last_delta"][:, :, None], d_pred[:, :, :-1]], 2)
    L_acc = masked_huber(a_pred, b["accel"], mask, cfg.huber_delta)
    err2 = (out["pos"] - b["pos"]) ** 2
    nll = 0.5 * (out["logvar"] + err2 / torch.exp(out["logvar"]).clamp(min=1e-6))
    L_nll = (nll.sum(-1) * mask).sum() / mask.sum().clamp(min=1)
    total = (cfg.w_pos * L_pos + cfg.w_end * L_end + cfg.w_delta * L_del +
             cfg.w_acc * L_acc + cfg.w_nll * L_nll)
    return total, dict(pos=L_pos.item(), end=L_end.item(), delta=L_del.item(),
                       acc=L_acc.item(), nll=L_nll.item())

def build_optimizers(model, cfg):
    if cfg.use_muon:   # Muon for 2-D hidden weights, AdamW for the rest (opt-in)
        from torch.optim import AdamW
        m2 = [p for n, p in model.named_parameters() if p.ndim >= 2 and "head" not in n]
        rest = [p for n, p in model.named_parameters() if not (p.ndim >= 2 and "head" not in n)]
        return [MuonLike(m2, cfg.lr), AdamW(rest, lr=cfg.lr, weight_decay=cfg.weight_decay)]
    return [torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)]

class MuonLike(torch.optim.Optimizer):
    """Minimal Newton-Schulz momentum optimizer for 2-D weights (ablation only)."""
    def __init__(self, params, lr): super().__init__(params, dict(lr=lr))
    @torch.no_grad()
    def step(self, closure=None):
        for g in self.param_groups:
            for p in g["params"]:
                if p.grad is None: continue
                st = self.state[p]
                m = st.setdefault("m", torch.zeros_like(p.grad))
                m.lerp_(p.grad, 0.05)
                x = m / (m.norm() + 1e-7)
                for _ in range(5):
                    A = x @ x.T; x = 3.4445 * x - 4.7750 * (A @ x) + 2.0315 * (A @ A @ x)
                p.add_(x, alpha=-g["lr"])

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.is_floating_point(): self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else: self.shadow[k] = v.detach().clone()

@contextmanager
def ema_weights(model, ema):
    live = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(ema.shadow)
    try: yield
    finally: model.load_state_dict(live)

## 6. Training loop
tqdm progress per batch; validation (EMA weights) every epoch with the official
RMSE; best-EMA checkpointing; early stopping (patience 6); history →
`outputs/history.json`.

In [ ]:
def train_hastnet(cfg):
    set_seed(cfg.seed)
    device = DEVICE
    plays = pd.concat([pd.read_csv(DATA_DIR / f"input_2023_w{w:02d}.csv").assign(week=w)
                       for w in sorted(set(cfg.train_weeks) | set(cfg.val_weeks) | set(cfg.test_weeks))],
                      ignore_index=True)
    plays["week"] = plays["week"].astype(int)
    # rebuild play index from PREP cache keys
    idx_df = pd.DataFrame([(w, g, p) for (w, g, p) in PREP.keys()], columns=["week", "game_id", "play_id"])

    tr = HASTDataset(idx_df, cfg, cfg.train_weeks, PREP, train=True)
    
    b = collate([tr[i] for i in range(8)])

    va = HASTDataset(idx_df, cfg, cfg.val_weeks, PREP, train=False)
    te = HASTDataset(idx_df, cfg, cfg.test_weeks, PREP, train=False)
    tr_loader, va_loader, te_loader = make_loader(tr, True), make_loader(va, False), make_loader(te, False)

    model = HASTNet(cfg).to(device)
    npar = sum(p.numel() for p in model.parameters())
    print(f"[train] params={npar:,} train={len(tr)} val={len(va)} test={len(te)}")
    opts = build_optimizers(model, cfg)
    total = max(1, len(tr_loader) * cfg.epochs)
    lam = lambda s: (s / cfg.warmup_steps if s < cfg.warmup_steps else
                     max(0.0, 0.5 * (1 + math.cos(math.pi * (s - cfg.warmup_steps) / max(1, total - cfg.warmup_steps)))))
    scheds = [LambdaLR(o, lam) for o in opts]
    ema = EMA(model, cfg.ema_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp)
    ac = torch.amp.autocast("cuda", dtype=torch.float16, enabled=cfg.amp)
    history = dict(train_loss=[], val_rmse=[], parts=[])
    best, bad = math.inf, 0
    for ep in range(cfg.epochs):
        model.train(); lsum = msum = 0.0; parts = []
        pbar = tqdm(tr_loader, desc=f"epoch {ep:02d}", leave=False)
        for b in pbar:
            b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in b.items()}
            for o in opts: o.zero_grad(set_to_none=True)
            with ac:
                out = model(b)
                loss, parts_d = hast_loss(out, b, cfg)
            scaler.scale(loss).backward()
            for o in opts:
                scaler.unscale_(o)
                nn.utils.clip_grad_norm_([p for g in o.param_groups for p in g["params"]], cfg.grad_clip)
            for o in opts: scaler.step(o)
            scaler.update()
            for s in scheds: s.step()
            ema.update(model)
            m = float((b["valid"] * b["to_pred"][..., None]).sum())
            lsum += loss.item() * m; msum += m; parts.append(parts_d)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        with ema_weights(model, ema):
            vrmse = eval_rmse(model, va_loader, cfg, device)
        history["train_loss"].append(lsum / max(1, msum)); history["val_rmse"].append(vrmse)
        history["parts"].append({k: float(np.mean([p[k] for p in parts])) for k in parts[0]})
        print(f"epoch {ep:02d} | train_loss={history['train_loss'][-1]:.4f} | val_rmse={vrmse:.4f}")
        if vrmse < best - 1e-4:
            best, bad = vrmse, 0
            torch.save(dict(model=model.state_dict(), ema=ema.shadow, config=asdict(cfg),
                            epoch=ep, val_rmse=vrmse), CKPT_DIR / f"{cfg.exp_name}_best.pt")
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"[train] early stop @ epoch {ep} (best {best:.4f})"); break
    with open(OUT_DIR / "history.json", "w") as f: json.dump(history, f, indent=2)
    with open(OUT_DIR / "config.json", "w") as f: json.dump(asdict(cfg), f, indent=2)
    return model, ema, history, te_loader

In [ ]:
def official_rmse(sq_sum, n): return float(math.sqrt(sq_sum / (2.0 * max(n, 1))))

@torch.no_grad()
def predict_pos(model, b):
    out = model(b)
    return out, out["pos"]

@torch.no_grad()
def eval_rmse(model, loader, cfg, device):
    model.eval(); sq = n = 0
    for b in loader:
        b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in b.items()}
        _, pos = predict_pos(model, b)
        m = (b["valid"] * b["to_pred"][..., None]).bool()
        sq += float(((pos - b["pos"]) ** 2).sum(-1)[m].sum()); n += int(m.sum())
    return official_rmse(sq, n)

@torch.no_grad()
def collect_test_stats(model, loader, cfg, device):
    """Per-play squared errors, horizon/role breakdowns and per-pair sigma calibration."""
    model.eval()
    plays_sq, plays_n = [], []
    hor_sq, hor_n = [], []
    role_sq, role_n = [], []
    sig_pred, sig_obs = [], []
    for b in loader:
        b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in b.items()}
        out = model(b)
        pos = out["pos"]
        m = (b["valid"] * b["to_pred"][..., None]).bool()          # (B,P,O)
        m_np = m.cpu().numpy()
        e2_np = ((pos - b["pos"]) ** 2).sum(-1).cpu().numpy()      # (B,P,O) squared radial error
        # per-pair predicted radial sigma and observed radial error:
        # both are 1-D chunks of the SAME length N_b -> safe to concatenate later
        sig_pred.append(np.sqrt(np.exp(out["logvar"].cpu().numpy()).mean(-1))[m_np])
        sig_obs.append(np.sqrt(e2_np)[m_np])
        for i in range(e2_np.shape[0]):
            Pi = int(b["p_mask"][i].sum()); Oi = int(b["o_mask"][i, 0].sum())
            
            # --- FIX: Move the mask to CPU and convert to numpy ---
            mi = m[i, :Pi, :Oi].cpu().numpy()  
            
            plays_sq.append(float(e2_np[i, :Pi, :Oi][mi].sum()))
            plays_n.append(int(mi.sum()))
            for o in range(Oi):
                mo = mi[:, o]
                if mo.any():
                    hor_sq.append(float(e2_np[i, :Pi, o][mo].sum()))
                    hor_n.append(int(mo.sum()))
            roles = b["roles"][i, :Pi].cpu().numpy()
            for p in range(Pi):
                mp = mi[p]
                if mp.any():
                    role_sq.append(float(e2_np[i, p, :Oi][mp].sum()))
                    role_n.append(int(mp.sum()))
    return dict(plays_sq=np.array(plays_sq), plays_n=np.array(plays_n),
                hor_sq=np.array(hor_sq), hor_n=np.array(hor_n),
                role_sq=np.array(role_sq), role_n=np.array(role_n),
                sig_pred=np.concatenate(sig_pred), sig_obs=np.concatenate(sig_obs))


def bootstrap_rmse(sq, n, rng, n_boot, block):
    point = official_rmse(sq.sum(), n.sum()); N = len(sq)
    boots = []
    for _ in range(n_boot):
        idx = np.concatenate([np.arange(s, min(s + block, N)) for s in
                              rng.integers(0, N, max(1, N // block))])
        idx = idx[idx < N]
        boots.append(official_rmse(sq[idx].sum(), n[idx].sum()))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return point, float(lo), float(hi)

@torch.no_grad()
def const_velocity_stats(loader, cfg):
    sq, n = [], []
    for b in loader:
        last, ldel = b["last_xy"], b["last_delta"]
        O = int(b["pos"].shape[2])
        t = (torch.arange(O).float() + 1)[None, None, :, None]
        pos = last[:, :, None, :] + ldel[:, :, None, :] * t
        m = (b["valid"] * b["to_pred"][..., None]).bool()
        e2 = ((pos - b["pos"]) ** 2).sum(-1)
        for i in range(e2.shape[0]):
            Pi = int(b["p_mask"][i].sum()); Oi = int(b["o_mask"][i, 0].sum()); mi = m[i, :Pi, :Oi]
            sq.append(float(e2[i, :Pi, :Oi][mi].sum())); n.append(int(mi.sum()))
    return np.array(sq), np.array(n)

## 7. Evaluation & statistics
Official RMSE on week 18, block-bootstrap 95% CIs over plays, constant-velocity
baseline, per-role and per-horizon breakdowns, σ-calibration.

In [ ]:
MODEL, EMA_M, HISTORY, TE_LOADER = train_hastnet(CFG)

rng = np.random.default_rng(CFG.seed)
stats = collect_test_stats(MODEL, TE_LOADER, CFG, DEVICE)
pt, lo, hi = bootstrap_rmse(stats["plays_sq"], stats["plays_n"], rng, CFG.n_bootstrap, CFG.bootstrap_block)
bsq, bn = const_velocity_stats(TE_LOADER, CFG)
bpt, blo, bhi = bootstrap_rmse(bsq, bn, rng, CFG.n_bootstrap, CFG.bootstrap_block)
results = {"HAST-Net (EMA)": dict(rmse=pt, ci_lo=lo, ci_hi=hi),
           "constant-velocity": dict(rmse=bpt, ci_lo=blo, ci_hi=bhi)}
# per-horizon curve with bootstrap CIs
maxO = int(stats["hor_n"].size and stats["hor_sq"].size and 48)
hor_bins = np.arange(0, 48, 4)
hor_curve = []
for i0 in range(len(hor_bins) - 1):
    sel = (stats["hor_sq"] >= 0)  # placeholder replaced below
hor_table = []
for o in range(0, 40, 4):
    sel = stats["hor_n"] > 0
    # per-horizon aggregation: hor arrays are per (play,horizon) pairs in order
    mask_o = np.zeros_like(stats["hor_sq"], bool)
    # hor arrays were appended per (play, o) in increasing o -> rebuild via counts
    # simpler: recompute per-horizon from stored per-play arrays is heavy; use stored pairs
    break
# per-horizon from stored pairs (hor_sq/hor_n were appended per (play,o))
hor_pairs = list(zip(stats["hor_sq"], stats["hor_n"]))
# rebuild horizon index: pairs were appended in o order per play -> recover via grouping
# (stored sequentially per play, o ascending) -> reconstruct using play lengths
hor_idx = np.concatenate([np.arange(int(n // max(1, n)) ) for n in []]) if False else None
# robust reconstruction: recompute per-horizon directly from test loader once
# Исправленный per-horizon RMSE (без двойного возведения в квадрат) + NaN для пустых бинов
import numpy as np, torch

def per_horizon_rmse_correct(model, loader, device, maxO=48, n_boot=200, seed=0):
    model.eval()
    sq = [[] for _ in range(maxO)]
    rng = np.random.default_rng(seed)
    with torch.no_grad():
        for b in loader:
            b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in b.items()}
            out = model(b)
            m = (b["valid"] * b["to_pred"][..., None]).bool().cpu().numpy()
            e2 = ((out["pos"] - b["pos"]) ** 2).sum(-1).cpu().numpy()   # radial sq per pair
            for o in range(min(maxO, e2.shape[2])):
                v = e2[:, :, o][m[:, :, o]]
                if v.size:
                    sq[o].append(v)
    rmse, lo, hi = [], [], []
    for o in range(maxO):
        if not sq[o]:
            rmse.append(np.nan); lo.append(np.nan); hi.append(np.nan); continue
        v = np.concatenate(sq[o])
        rmse.append(float(np.sqrt(v.mean() / 2.0)))                 # per-coordinate RMSE
        boots = [np.sqrt(v[rng.integers(0, v.size, v.size)].mean() / 2.0) for _ in range(n_boot)]
        lo.append(np.percentile(boots, 2.5)); hi.append(np.percentile(boots, 97.5))
    return np.array(rmse), np.array(lo), np.array(hi)   # при plot использовать NaN-маску

hor_err = per_horizon_rmse_correct(MODEL, TE_LOADER, DEVICE)
hor_mean = [official_rmse(float((v ** 2).sum()), v.size) for v in hor_err]
hor_lo, hor_hi = [], []
for v in hor_err:
    boots = [official_rmse(float((v[rng.integers(0, v.size, v.size)] ** 2).sum()), v.size) for _ in range(200)]
    lo_, hi_ = np.percentile(boots, [2.5, 97.5]); hor_lo.append(lo_); hor_hi.append(hi_)
print(json.dumps(results, indent=2))
with open(OUT_DIR / "test_results.json", "w") as f:
    json.dump(dict(results=results, horizon_rmse=hor_mean,
                   horizon_ci=[hor_lo, hor_hi]), f, indent=2)

## 8. Visualizations
Training curves, RMSE comparison with 95% CIs, RMSE-vs-horizon with CI band,
per-role boxplots, error percentiles, σ-calibration, trajectory examples.

In [ ]:
# ============================ CELL 17: figures ============================
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.grid": True, "grid.alpha": .3})

# (1) training curves
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(HISTORY["train_loss"], label="train loss")
ax[0].plot(HISTORY["val_rmse"], label="val RMSE (EMA)")
ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].set_title("Training / validation")
parts = pd.DataFrame(HISTORY["parts"])
parts.plot(ax=ax[1], title="loss components")
ax[1].set_xlabel("epoch")
fig.tight_layout(); fig.savefig(FIG_DIR / "training_curves.png"); plt.close(fig)

# (2) RMSE comparison with 95% CI
names = list(results); pts = np.array([results[n]["rmse"] for n in names])
los = np.array([results[n]["ci_lo"] for n in names]); his = np.array([results[n]["ci_hi"] for n in names])
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(names, pts, yerr=[pts - los, his - pts], capsize=6, color=["#4c72b0", "#dd8452"])
ax.set_ylabel("test RMSE (yards)"); ax.set_title("Test RMSE with 95% CI (block bootstrap)")
fig.tight_layout(); fig.savefig(FIG_DIR / "rmse_comparison.png"); plt.close(fig)

# (3) RMSE vs horizon with CI band
xs = np.arange(len(hor_mean))
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(xs, hor_mean, marker="o", label="HAST-Net")
ax.fill_between(xs, hor_lo, hor_hi, alpha=.2)
ax.set_xlabel("future frame o"); ax.set_ylabel("RMSE(o) (yards)")
ax.set_title("Error growth over horizon (95% CI band)"); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "rmse_vs_horizon.png"); plt.close(fig)

# (4) error percentiles (heavy tail)
obs = np.concatenate([v for v in hor_err])
qs = np.linspace(1, 99, 99)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(qs, np.percentile(np.sqrt(obs / 2), qs), marker=".", ls="")
ax.set_xlabel("percentile"); ax.set_ylabel("radial error (yards)")
ax.set_title("Error percentiles"); fig.tight_layout()
fig.savefig(FIG_DIR / "error_percentiles.png"); plt.close(fig)

# (5) sigma calibration (predicted sigma vs observed error, binned)
sp, so = stats["sig_pred"], stats["sig_obs"]
bins = np.quantile(sp, np.linspace(0, 1, 11))
bx = [0.5 * (bins[i] + bins[i + 1]) for i in range(10)]
by = [so[(sp >= bins[i]) & (sp < bins[i + 1])].mean() for i in range(10)]
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(bx, by, "o-", label="observed mean |err|")
ax.plot([0, max(bx)], [0, max(bx)], "k--", label="ideal")
ax.set_xlabel("predicted sigma"); ax.set_ylabel("observed error")
ax.set_title("Sigma calibration"); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "sigma_calibration.png"); plt.close(fig)

# (6) trajectory examples
@torch.no_grad()
def traj_examples(model, loader, device, n=4):
    model.eval(); b = next(iter(loader))
    b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in b.items()}
    _, pos = predict_pos(model, b)
    pos = pos.cpu().numpy(); true = b["pos"].cpu().numpy()
    m = (b["valid"] * b["to_pred"][..., None]).bool().cpu().numpy()
    fig, axes = plt.subplots(1, n, figsize=(4.3 * n, 4.3), squeeze=False)
    for ax, i in zip(axes[0], range(n)):
        Pi = int(b["p_mask"][i].sum()); Oi = int(b["o_mask"][i, 0].sum())
        for p in range(Pi):
            if m[i, p, 0]:
                ax.plot(true[i, p, :Oi, 0], true[i, p, :Oi, 1], "-", lw=1.4)
                ax.plot(pos[i, p, :Oi, 0], pos[i, p, :Oi, 1], "--", lw=1.1)
        ax.set_aspect("equal"); ax.set_title(f"play {i}")
    fig.suptitle("true (solid) vs predicted (dashed)")
    fig.tight_layout(); fig.savefig(FIG_DIR / "trajectories.png"); plt.close(fig)
traj_examples(MODEL, TE_LOADER, DEVICE)
print("figures saved to", FIG_DIR)

## 9. Package artifacts
`checkpoints/`, `figures/`, `outputs/` → `/kaggle/working/hastnet_artifacts.zip`.

In [ ]:
# ============================ CELL 18: zip artifacts ============================
zip_path = WORK / "hastnet_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for d in (CKPT_DIR, FIG_DIR, OUT_DIR):
        for root, _, files in os.walk(d):
            for fn in files:
                p = Path(root) / fn
                z.write(p, p.relative_to(WORK))
print("archive:", zip_path, f"({zip_path.stat().st_size/1e6:.1f} MB)")
for root, _, files in os.walk(WORK):
    for fn in sorted(files): print(" ", Path(root) / fn)

## 10. Reproducibility checklist
- `set_seed(42)` fixes Python/NumPy/Torch; cuDNN deterministic; DataLoader
  generator + `worker_init_fn` seed the augmentation RNGs.
- Config dumped to `outputs/config.json`; checkpoint stores config + EMA weights
  + epoch + val RMSE.
- Same week split & evaluation harness as the reproduced 1st/3rd/5th-place
  baselines → numbers directly comparable.
- Local verification: `python train.py evaluate --exp-name hastnet` reproduces
  the test RMSE within ~1e-3.
- Download `hastnet_artifacts.zip` from the Output pane and unpack into the
  local repository (`checkpoints/`, `figures/`, `outputs/` merge automatically).

## 11. Hyperparameters Tuning

In [ ]:
# ============ CELL T1: hyperparameter tuning (3 configurations) ============
# Protocol: same seed (42), same week split (w01-15 / w16-17 / w18), same harness;
# selection by best EMA val RMSE; winner re-evaluated on test with block-bootstrap CI.
# Runtime control: 12 epochs + patience 4 per config; PREP cache reused (no CSV re-read).
import shutil

TUNE_DIR = WORK / "tuning"
for _sub in ("checkpoints", "figures", "outputs"):
    (TUNE_DIR / _sub).mkdir(parents=True, exist_ok=True)

# snapshot baseline outputs (train_hastnet overwrites outputs/history.json & config.json)
shutil.copy(OUT_DIR / "history.json", TUNE_DIR / "outputs" / "baseline_history.json")
shutil.copy(OUT_DIR / "config.json",  TUNE_DIR / "outputs" / "baseline_config.json")

TUNING_CONFIGS = [
    # T1: capacity + horizon resolution (architecture change)
    dict(name="tune_capacity",
         overrides=dict(d_model=192, n_heads=8, n_control=12, rnn_hidden=128,
                        epochs=12, patience=4)),
    # T2: stronger regularization + augmentation (mild overfit / invariance)
    dict(name="tune_regaug",
         overrides=dict(dropout=0.2, weight_decay=0.05,
                        p_rot=0.7, rot_deg=15.0, p_shift=0.7, shift_yd=4.0, p_fshift=0.4,
                        epochs=12, patience=4)),
    # T3: loss re-balancing for heavy tail / endpoint anchor (EDA: landing anchor, p99 tail)
    dict(name="tune_loss_tail",
         overrides=dict(w_end=1.0, w_delta=0.7, w_nll=0.3, huber_delta=0.5, lr=4e-4,
                        epochs=12, patience=4))
]

idx_df = pd.DataFrame([(w, g, p) for (w, g, p) in PREP.keys()],
                      columns=["week", "game_id", "play_id"])

tune_results = []
for spec in TUNING_CONFIGS:
    cfg_t = Config(**{**asdict(CFG), **spec["overrides"]})
    cfg_t.exp_name = spec["name"]
    t0 = time.time()
    model_t, ema_t, hist_t, _ = train_hastnet(cfg_t)          # reuses PREP, tqdm, EMA, early stop
    dt = time.time() - t0
    best_ep  = int(np.argmin(hist_t["val_rmse"]))
    best_val = float(min(hist_t["val_rmse"]))
    shutil.copy(CKPT_DIR / f"{spec['name']}_best.pt", TUNE_DIR / "checkpoints")
    shutil.copy(OUT_DIR / "history.json", TUNE_DIR / "outputs" / f"{spec['name']}_history.json")
    shutil.copy(OUT_DIR / "config.json",  TUNE_DIR / "outputs" / f"{spec['name']}_config.json")
    tune_results.append(dict(name=spec["name"], overrides=spec["overrides"],
                             n_params=int(sum(p.numel() for p in model_t.parameters())),
                             best_epoch=best_ep, best_val_rmse=best_val,
                             final_val_rmse=float(hist_t["val_rmse"][-1]),
                             time_sec=round(dt, 1), history=hist_t))
    print(f"[tuning] {spec['name']}: best val RMSE {best_val:.4f} @ epoch {best_ep} ({dt/60:.1f} min)")

best = min(tune_results, key=lambda r: r["best_val_rmse"])
base, base_cv = results["HAST-Net (EMA)"], results["constant-velocity"]
print(f"[tuning] best config: {best['name']} (val {best['best_val_rmse']:.4f}) "
      f"vs baseline val {min(HISTORY['val_rmse']):.4f}")

# ---- test evaluation of the winning config (same harness as baseline) ----
cfg_b = Config(**{**asdict(CFG), **best["overrides"]}); cfg_b.exp_name = best["name"]
te_ds     = HASTDataset(idx_df, cfg_b, cfg_b.test_weeks, PREP, train=False)
te_loader = make_loader(te_ds, False)
model_b   = HASTNet(cfg_b).to(DEVICE)
ck = torch.load(TUNE_DIR / "checkpoints" / f"{best['name']}_best.pt", map_location=DEVICE)
model_b.load_state_dict(ck["ema"]); model_b.eval()   # EMA weights, baseline protocol
rng_t   = np.random.default_rng(cfg_b.seed)
stats_b = collect_test_stats(model_b, te_loader, cfg_b, DEVICE)
ptb, lob, hib = bootstrap_rmse(stats_b["plays_sq"], stats_b["plays_n"],
                               rng_t, cfg_b.n_bootstrap, cfg_b.bootstrap_block)
print(f"[tuning] test RMSE {ptb:.4f} [{lob:.4f}, {hib:.4f}] | baseline HAST-Net "
      f"{base['rmse']:.4f} [{base['ci_lo']:.4f}, {base['ci_hi']:.4f}] | const-vel {base_cv['rmse']:.4f}")

# ---- tuning figures ----
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(HISTORY["val_rmse"], lw=2, color="#888888",
        label=f"baseline (best {min(HISTORY['val_rmse']):.3f})")
for r in tune_results:
    ax.plot(r["history"]["val_rmse"], label=f"{r['name']} (best {r['best_val_rmse']:.3f})")
ax.set_xlabel("epoch"); ax.set_ylabel("val RMSE (yards)")
ax.set_title("Hyperparameter tuning: validation RMSE curves"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(TUNE_DIR / "figures" / "tuning_val_curves.png"); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.5))
names = ["baseline"] + [r["name"] for r in tune_results]
vals  = [min(HISTORY["val_rmse"])] + [r["best_val_rmse"] for r in tune_results]
bars  = ax.bar(names, vals, color=["#888888", "#4c72b0", "#55a868", "#c44e52"])
ax.axhline(min(HISTORY["val_rmse"]), ls="--", lw=1, color="#888888")
for bb, v in zip(bars, vals):
    ax.text(bb.get_x() + bb.get_width() / 2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("best val RMSE (yards)")
ax.set_title("Hyperparameter tuning: best val RMSE per configuration")
fig.tight_layout(); fig.savefig(TUNE_DIR / "figures" / "tuning_comparison.png"); plt.close(fig)

n_base = int(sum(p.numel() for p in HASTNet(CFG).parameters()))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter([n_base / 1e6], [min(HISTORY["val_rmse"])], s=60, color="#888888")
ax.annotate("baseline", (n_base / 1e6, min(HISTORY["val_rmse"])), xytext=(6, 4),
            textcoords="offset points", fontsize=8)
for r in tune_results:
    ax.scatter([r["n_params"] / 1e6], [r["best_val_rmse"]], s=60)
    ax.annotate(r["name"], (r["n_params"] / 1e6, r["best_val_rmse"]), xytext=(6, 4),
                textcoords="offset points", fontsize=8)
ax.set_xlabel("params (M)"); ax.set_ylabel("best val RMSE (yards)")
ax.set_title("Capacity vs quality"); fig.tight_layout()
fig.savefig(TUNE_DIR / "figures" / "tuning_params_vs_val.png"); plt.close(fig)

# ---- tuning summary JSON + stable copy of the winning checkpoint ----
tuning_summary = dict(
    protocol=dict(seed=CFG.seed, split="train w01-15 / val w16-17 / test w18",
                  selection="best EMA val RMSE", epochs_per_config=12, patience=4),
    baseline=dict(exp_name="hastnet", n_params=n_base,
                  best_val_rmse=float(min(HISTORY["val_rmse"])),
                  test_rmse=base["rmse"], test_ci=[base["ci_lo"], base["ci_hi"]],
                  const_velocity_rmse=base_cv["rmse"]),
    configs=[{k: r[k] for k in ("name", "overrides", "n_params", "best_epoch",
                                "best_val_rmse", "final_val_rmse", "time_sec")}
             for r in tune_results],
    best_config=best["name"], best_test_rmse=ptb, best_test_ci=[lob, hib],
)
with open(TUNE_DIR / "outputs" / "tuning_results.json", "w") as f:
    json.dump(tuning_summary, f, indent=2)
shutil.copy(TUNE_DIR / "checkpoints" / f"{best['name']}_best.pt",
            TUNE_DIR / "checkpoints" / "tuning_best_overall.pt")

# restore baseline outputs overwritten during tuning
shutil.copy(TUNE_DIR / "outputs" / "baseline_history.json", OUT_DIR / "history.json")
shutil.copy(TUNE_DIR / "outputs" / "baseline_config.json",  OUT_DIR / "config.json")

# ---- separate zip archive for tuning artifacts ----
tune_zip = WORK / "hastnet_tuning_artifacts.zip"
with zipfile.ZipFile(tune_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(TUNE_DIR):
        for fn in files:
            p = Path(root) / fn
            z.write(p, p.relative_to(WORK))
print("tuning archive:", tune_zip, f"({tune_zip.stat().st_size/1e6:.1f} MB)")
for root, _, files in os.walk(TUNE_DIR):
    for fn in sorted(files):
        print("  ", Path(root) / fn)